# NB20 — CFTR Ensemble + Threshold Refinement

**TEKNOFEST Sağlıkta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

## Amaç

NB19'da iki kritik bulgu çıktı:
- **S0_MASTER-only**: LOO-MCC=0.6552, boot_mean=0.6915 → LOO metriğinde iyi, FP=3
- **S0c_COMBINED**: LOO-MCC=0.6436, boot_mean=0.8629 → Boot metriğinde iyi, FP=0
- **Prior-shift anomalisi**: COMBINED'da mcc_raw==mcc_prior — threshold değişmedi

Bu notebook üç soruyu yanıtlar:

### Experiment 1 — Soft Ensemble
`p_ens = 0.5 * p_S0_adjusted + 0.5 * p_COMBINED_adjusted`

LOO-MCC ve boot_mean'i **aynı anda** optimize etmek mümkün mü?

### Experiment 2 — Prior-Shift Debug
COMBINED'da neden prior-shift etkisiz? Posterior histogramı analiz et.

### Experiment 3 — Robust Threshold
Tek resample yerine N=50 bootstrap ortalaması → threshold seçimini gürültüden arındır.

**Birincil metrik**: LOO-CV MCC (n=21 benign → bootstrap CI anlamsız)  
**İkincil metrik**: boot_mean @%80/20 (yarışma final dağılımı simülasyonu)

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay
)
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
import lightgbm as lgb

LGBM_FIXED = {
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbosity': -1,
    'random_state': SEED,
    'objective': 'binary',
    'class_weight': 'balanced',
}

np.random.seed(SEED)

PANEL             = "CFTR"
HIGH_MISSING_THR  = 0.50
FINAL_BENIGN_FRAC = 0.80
N_BOOT            = 50
BOOT_SEED         = SEED
PI_TEST           = 0.20
N_ROBUST          = 50  # robust threshold bootstrap sayisi

DATA_DIR    = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v10_cftr_ensemble")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SEED         : {SEED}")
print(f"RESULTS_DIR  : {RESULTS_DIR}")

PROJECT_ROOT : /Users/tefe/teknofest_model/teknofest_model
SEED         : 42
RESULTS_DIR  : /Users/tefe/teknofest_model/teknofest_model/results/v10_cftr_ensemble


In [2]:
# Cell 2: Veri Yükleme + Sütun Temizliği
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master   = load_panel("MASTER")
kanser   = load_panel("KANSER")
pah      = load_panel("PAH")
cftr_raw = load_panel("CFTR")

feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

def drop_exact_duplicates(df, ref_df, name):
    """df'den ref_df ile birebir-ayni satirlari drop et."""
    ref_index = {}
    for _, row in ref_df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        ref_index[key] = row[TARGET]
    drop_idx = []
    for idx, row in df.iterrows():
        key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                                     for v in row[feature_cols_all].values)
        if key in ref_index and ref_index[key] == row[TARGET]:
            drop_idx.append(idx)
    df_clean = df.drop(index=drop_idx).reset_index(drop=True)
    print(f"{name}: {df.shape[0]} -> {df_clean.shape[0]} (drop={len(drop_idx)} birebir-ayni)")
    return df_clean

cftr         = drop_exact_duplicates(cftr_raw, master, "CFTR")
kanser_clean = drop_exact_duplicates(kanser, cftr_raw, "KANSER")
pah_clean    = drop_exact_duplicates(pah, cftr_raw, "PAH")

combined = pd.concat([master, kanser_clean, pah_clean], ignore_index=True)

print(f"\nMASTER  : {master.shape}, label: {master[TARGET].value_counts().to_dict()}")
print(f"KANSER  : {kanser_clean.shape}, label: {kanser_clean[TARGET].value_counts().to_dict()}")
print(f"PAH     : {pah_clean.shape}, label: {pah_clean[TARGET].value_counts().to_dict()}")
print(f"CFTR    : {cftr.shape}, label: {cftr[TARGET].value_counts().to_dict()}")
print(f"COMBINED: {combined.shape}, label: {combined[TARGET].value_counts().to_dict()}")

constant_cols     = CR.get_constant_cols(master[feature_cols_all])
dup_pairs         = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop          = sorted({b for (a, b) in dup_pairs})
drop_cols         = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE          = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE     = [c for c in base_feature_cols if c not in CAT_LIKE]

print(f"\nSutun temizligi: drop {len(drop_cols)} -> {len(base_feature_cols)} feature")
print(f"  ({len(NUM_COLS_BASE)} sayisal + {len(CAT_LIKE)} kategorik)")

CFTR: 111 -> 111 (drop=0 birebir-ayni)
KANSER: 388 -> 388 (drop=0 birebir-ayni)
PAH: 372 -> 372 (drop=0 birebir-ayni)

MASTER  : (2931, 353), label: {1: 2149, 0: 782}
KANSER  : (388, 353), label: {1: 268, 0: 120}
PAH     : (372, 353), label: {1: 310, 0: 62}
CFTR    : (111, 353), label: {1: 90, 0: 21}
COMBINED: (3691, 353), label: {1: 2727, 0: 964}

Sutun temizligi: drop 63 -> 288 feature
  (281 sayisal + 7 kategorik)


In [3]:
# Cell 3: FE + M3 Preprocessing
AA_UNK      = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

_B62_RAW = """A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4"""
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for _ri, _line in enumerate(_B62_RAW.strip().split("\n")):
    _toks = _line.split()
    _row_aa = _toks[0][0]
    _vals = [_toks[0][1:]] + _toks[1:]
    for _ci, _tok in enumerate(_vals):
        _col_aa = _ORDER[_ri + _ci]
        _v = int(_tok[1:] if _tok[0].isalpha() else _tok)
        _B62[(_row_aa, _col_aa)] = _v; _B62[(_col_aa, _row_aa)] = _v
def blosum62(a, b): return _B62.get((a, b), 0)

def add_fe(df):
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v): return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]

def detect_log_cols(train_df):
    cand = []
    for c in NUM_COLS_BASE:
        s = pd.to_numeric(train_df[c], errors="coerce").dropna()
        if len(s) < 10: continue
        if s.min() >= 0 and s.max() > 1.0 and s.skew() > 2.0:
            cand.append(c)
    return cand

def fit_preprocessor(train_df_raw):
    tr = add_fe(train_df_raw)
    log_cols = detect_log_cols(tr)
    num_cols = NUM_COLS_BASE + FE_NEW_COLS + [f"{c}__log" for c in log_cols]
    for c in log_cols:
        tr[f"{c}__log"] = np.log1p(pd.to_numeric(tr[c], errors="coerce").clip(lower=0))
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THR].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {"log_cols": log_cols, "num_cols": num_cols, "cat_cols": CAT_LIKE,
            "flag_source": flag_source, "median": median}

def transform_X(df_raw, pp):
    df = add_fe(df_raw)
    for c in pp["log_cols"]:
        df[f"{c}__log"] = np.log1p(pd.to_numeric(df[c], errors="coerce").clip(lower=0))
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"][c]).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df[c].isna().astype(int).values
    return out, list(pp["cat_cols"])

pp_master   = fit_preprocessor(master)
X_master_df, cat_cols = transform_X(master, pp_master)
y_master    = master[TARGET].values

pp_combined = fit_preprocessor(combined)
X_combined_df, _ = transform_X(combined, pp_combined)
y_combined  = combined[TARGET].values

X_cftr_master_df,   _ = transform_X(cftr, pp_master)
X_cftr_combined_df, _ = transform_X(cftr, pp_combined)
y_cftr = cftr[TARGET].values

pi_train_master   = float(y_master.mean())
pi_train_combined = float(y_combined.mean())

print(f"pp_master  : {X_master_df.shape[1]} feature, log={len(pp_master['log_cols'])}, flag={len(pp_master['flag_source'])}")
print(f"pp_combined: {X_combined_df.shape[1]} feature, log={len(pp_combined['log_cols'])}, flag={len(pp_combined['flag_source'])}")
print(f"CFTR: {X_cftr_master_df.shape} (master) / {X_cftr_combined_df.shape} (combined)")
print(f"pi_train_master={pi_train_master:.4f}, pi_train_combined={pi_train_combined:.4f}")

pp_master  : 432 feature, log=0, flag=140
pp_combined: 432 feature, log=0, flag=140
CFTR: (111, 432) (master) / (111, 432) (combined)
pi_train_master=0.7332, pi_train_combined=0.7388


In [4]:
# Cell 4: Değerlendirme Altyapısı (NB19 bazı + yeni select_threshold_robust)

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0: return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED); f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020(y, prob):
    """Tek resample'da F1-max threshold (NB19 referans, karsilastirma icin)."""
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y, prob, rng)
    best, best_thr = -1.0, 0.5
    for thr in np.arange(0.05, 0.95, 0.01):
        f = _f1_pos(yb, (pb >= thr).astype(int))
        if f > best: best, best_thr = f, thr
    return float(best_thr)

def select_threshold_robust(y, prob, n=N_ROBUST):
    """N=50 bootstrap uzerinden ortalama threshold — gürültüden arindirilmis."""
    rng = np.random.RandomState(BOOT_SEED)
    thresholds = []
    for _ in range(n):
        yb, pb = _resample_8020(y, prob, rng)
        best, best_thr = -1.0, 0.5
        for thr in np.arange(0.05, 0.95, 0.01):
            f = _f1_pos(yb, (pb >= thr).astype(int))
            if f > best: best, best_thr = f, thr
        thresholds.append(best_thr)
    return float(np.mean(thresholds))

def loo_metrics_with_thr(y_true, prob, thr):
    """Verilen threshold ile LOO metrikleri hesapla."""
    y_pred = (prob >= thr).astype(int)
    mcc  = matthews_corrcoef(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.0
    f1   = _f1_pos(y_true, y_pred)
    auc  = roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else 0.0
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_true, prob, thr)
    cm   = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {"mcc": mcc, "f1": f1, "auc": auc, "precision": prec, "recall": rec,
            "thr": thr, "boot8020": boot, "y_pred": y_pred, "y_true": y_true,
            "prob": prob, "cm": cm}

def eval_strategy(y_true, prob_raw, pi_train, use_prior_shift=True, use_robust_thr=True):
    """Strateji icin tam degerlendirme: prior-shift + threshold secimi + metrikler."""
    if use_prior_shift:
        prob = adjust_prior_shift(prob_raw, pi_train)
    else:
        prob = prob_raw
    if use_robust_thr:
        thr = select_threshold_robust(y_true, prob)
    else:
        thr = select_threshold_8020(y_true, prob)
    return loo_metrics_with_thr(y_true, prob, thr)

print("Degerlendirme altyapisi hazir.")
print("  - select_threshold_8020  : tek resample (NB19 referans)")
print("  - select_threshold_robust: N=50 bootstrap ortalamasi (YENİ)")
print("  - eval_strategy          : prior-shift + threshold + metrikler")

Degerlendirme altyapisi hazir.
  - select_threshold_8020  : tek resample (NB19 referans)
  - select_threshold_robust: N=50 bootstrap ortalamasi (YENİ)
  - eval_strategy          : prior-shift + threshold + metrikler


In [5]:
# Cell 5: Model Egitimi — S0 (MASTER) ve S0c (COMBINED)
def _le_encode(X_df):
    Xn = X_df.copy()
    for c in cat_cols:
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c].astype(str))
    return Xn.astype(float)

def _lgbm_classifier():
    return lgb.LGBMClassifier(**{**LGBM_FIXED, "n_estimators": 200,
                                  "num_leaves": 31, "learning_rate": 0.05})

X_master_le   = _le_encode(X_master_df)
X_combined_le = _le_encode(X_combined_df)
X_cftr_m_le   = _le_encode(X_cftr_master_df)
X_cftr_c_le   = _le_encode(X_cftr_combined_df)

print("S0_MASTER-only egitiliyor...")
m_s0 = _lgbm_classifier()
m_s0.fit(X_master_le, y_master)
p_s0_cftr = m_s0.predict_proba(X_cftr_m_le)[:, 1]
print(f"  p_s0_cftr: min={p_s0_cftr.min():.4f}, max={p_s0_cftr.max():.4f}, mean={p_s0_cftr.mean():.4f}")

print("S0c_COMBINED egitiliyor...")
m_s0c = _lgbm_classifier()
m_s0c.fit(X_combined_le, y_combined)
p_s0c_cftr = m_s0c.predict_proba(X_cftr_c_le)[:, 1]
print(f"  p_s0c_cftr: min={p_s0c_cftr.min():.4f}, max={p_s0c_cftr.max():.4f}, mean={p_s0c_cftr.mean():.4f}")

# Prior-shift uygulanmis posteriorlar
p_s0_adjusted  = adjust_prior_shift(p_s0_cftr, pi_train_master)
p_s0c_adjusted = adjust_prior_shift(p_s0c_cftr, pi_train_combined)
p_ensemble     = 0.5 * p_s0_adjusted + 0.5 * p_s0c_adjusted

print(f"\nPrior-shift sonrasi:")
print(f"  p_s0_adj : min={p_s0_adjusted.min():.4f}, max={p_s0_adjusted.max():.4f}, mean={p_s0_adjusted.mean():.4f}")
print(f"  p_s0c_adj: min={p_s0c_adjusted.min():.4f}, max={p_s0c_adjusted.max():.4f}, mean={p_s0c_adjusted.mean():.4f}")
print(f"  p_ens    : min={p_ensemble.min():.4f}, max={p_ensemble.max():.4f}, mean={p_ensemble.mean():.4f}")
print("\nModeller hazir.")

S0_MASTER-only egitiliyor...
  p_s0_cftr: min=0.0143, max=0.9968, mean=0.6840
S0c_COMBINED egitiliyor...
  p_s0c_cftr: min=0.0157, max=0.9959, mean=0.6748

Prior-shift sonrasi:
  p_s0_adj : min=0.0013, max=0.9664, mean=0.3555
  p_s0c_adj: min=0.0014, max=0.9558, mean=0.3285
  p_ens    : min=0.0014, max=0.9472, mean=0.3420

Modeller hazir.


In [6]:
# Cell 6: Experiment 1 — Soft Ensemble
# p_ens = 0.5 * p_s0_adjusted + 0.5 * p_s0c_adjusted
# Prior-shift zaten uygulandigi icin eval_strategy'de use_prior_shift=False

print("="*70)
print("EXP 1: Soft Ensemble (p_ens = 0.5*p_S0_adj + 0.5*p_COMBINED_adj)")
print("="*70)

strategies_exp1 = {
    "S0_MASTER": (p_s0_adjusted,  False),   # prior-shift zaten uygulandi
    "S0c_COMBINED": (p_s0c_adjusted, False),
    "Ensemble": (p_ensemble, False),
}

exp1_results = {}
for name, (prob, use_ps) in strategies_exp1.items():
    thr = select_threshold_8020(y_cftr, prob)  # tek resample (NB19 referans)
    res = loo_metrics_with_thr(y_cftr, prob, thr)
    exp1_results[name] = res
    cm = res["cm"]
    print(f"\n{name}:")
    print(f"  thr={thr:.3f}  MCC={res['mcc']:.4f}  F1={res['f1']:.4f}  AUC={res['auc']:.4f}")
    print(f"  Precision={res['precision']:.4f}  Recall={res['recall']:.4f}")
    print(f"  Boot-mean={res['boot8020']['mean']:.4f} ± {res['boot8020']['std']:.4f}  "
          f"[{res['boot8020']['lo']:.3f}–{res['boot8020']['hi']:.3f}]")
    print(f"  CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

print("\n" + "="*70)
print("EXP 1 TABLO OZETI:")
print(f"{'Strateji':<15} {'MCC':>7} {'Boot-mean':>10} {'Boot-std':>9} {'TN':>4} {'FP':>4} {'FN':>4} {'TP':>4}")
print("-"*65)
for name, res in exp1_results.items():
    cm = res["cm"]
    print(f"{name:<15} {res['mcc']:>7.4f} {res['boot8020']['mean']:>10.4f} "
          f"{res['boot8020']['std']:>9.4f} {cm[0,0]:>4} {cm[0,1]:>4} {cm[1,0]:>4} {cm[1,1]:>4}")
print("="*70)

EXP 1: Soft Ensemble (p_ens = 0.5*p_S0_adj + 0.5*p_COMBINED_adj)

S0_MASTER:
  thr=0.080  MCC=0.6552  F1=0.9186  AUC=0.9423
  Precision=0.9634  Recall=0.8778
  Boot-mean=0.6915 ± 0.0877  [0.545–0.769]
  CM: TN=18, FP=3, FN=11, TP=79

S0c_COMBINED:
  thr=0.150  MCC=0.6436  F1=0.8820  AUC=0.9339
  Precision=1.0000  Recall=0.7889
  Boot-mean=0.8629 ± 0.1327  [0.571–1.000]
  CM: TN=21, FP=0, FN=19, TP=71

Ensemble:
  thr=0.180  MCC=0.5961  F1=0.8535  AUC=0.9407
  Precision=1.0000  Recall=0.7444
  Boot-mean=0.8221 ± 0.1399  [0.571–1.000]
  CM: TN=21, FP=0, FN=23, TP=67

EXP 1 TABLO OZETI:
Strateji            MCC  Boot-mean  Boot-std   TN   FP   FN   TP
-----------------------------------------------------------------
S0_MASTER        0.6552     0.6915    0.0877   18    3   11   79
S0c_COMBINED     0.6436     0.8629    0.1327   21    0   19   71
Ensemble         0.5961     0.8221    0.1399   21    0   23   67


In [7]:
# Cell 7: Experiment 2 — Prior-Shift Debug (Histogram Analizi)
# COMBINED'da neden prior-shift etkisiz?

print("="*70)
print("EXP 2: Prior-Shift Debug — Posterior Histogram Analizi")
print("="*70)

# Threshold karsilastirmasi
thr_s0_raw   = select_threshold_8020(y_cftr, p_s0_cftr)
thr_s0_adj   = select_threshold_8020(y_cftr, p_s0_adjusted)
thr_s0c_raw  = select_threshold_8020(y_cftr, p_s0c_cftr)
thr_s0c_adj  = select_threshold_8020(y_cftr, p_s0c_adjusted)

print(f"\nS0_MASTER  : thr_raw={thr_s0_raw:.3f}  ->  thr_prior={thr_s0_adj:.3f}  "
      f"(delta={thr_s0_adj - thr_s0_raw:+.3f})")
print(f"S0c_COMBINED: thr_raw={thr_s0c_raw:.3f}  ->  thr_prior={thr_s0c_adj:.3f}  "
      f"(delta={thr_s0c_adj - thr_s0c_raw:+.3f})")

# Posterior istatistikleri
mask_benign = y_cftr == 0
mask_patho  = y_cftr == 1

print(f"\nRaw posterior (S0_MASTER):")
print(f"  Benign  — mean={p_s0_cftr[mask_benign].mean():.4f}, "
      f"std={p_s0_cftr[mask_benign].std():.4f}, "
      f"<0.5: {(p_s0_cftr[mask_benign] < 0.5).sum()}/{mask_benign.sum()}")
print(f"  Pathogenic — mean={p_s0_cftr[mask_patho].mean():.4f}, "
      f"std={p_s0_cftr[mask_patho].std():.4f}, "
      f">=0.5: {(p_s0_cftr[mask_patho] >= 0.5).sum()}/{mask_patho.sum()}")

print(f"\nRaw posterior (S0c_COMBINED):")
print(f"  Benign  — mean={p_s0c_cftr[mask_benign].mean():.4f}, "
      f"std={p_s0c_cftr[mask_benign].std():.4f}, "
      f"<0.5: {(p_s0c_cftr[mask_benign] < 0.5).sum()}/{mask_benign.sum()}")
print(f"  Pathogenic — mean={p_s0c_cftr[mask_patho].mean():.4f}, "
      f"std={p_s0c_cftr[mask_patho].std():.4f}, "
      f">=0.5: {(p_s0c_cftr[mask_patho] >= 0.5).sum()}/{mask_patho.sum()}")

print(f"\nPrior-shift adjusted (S0c_COMBINED):")
print(f"  Benign  — mean={p_s0c_adjusted[mask_benign].mean():.4f}, "
      f"std={p_s0c_adjusted[mask_benign].std():.4f}")
print(f"  Pathogenic — mean={p_s0c_adjusted[mask_patho].mean():.4f}, "
      f"std={p_s0c_adjusted[mask_patho].std():.4f}")

# 2x2 histogram
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("EXP 2 — Prior-Shift Debug: Posterior Histogramlari\n"
             "(kirmizi=benign, mavi=pathogenic)", fontweight="bold", fontsize=12)

bins = np.linspace(0, 1, 30)
panel_data = [
    (axes[0,0], p_s0_cftr,      "S0_MASTER (raw)"),
    (axes[0,1], p_s0_adjusted,  "S0_MASTER (prior-shift)"),
    (axes[1,0], p_s0c_cftr,     "S0c_COMBINED (raw)"),
    (axes[1,1], p_s0c_adjusted, "S0c_COMBINED (prior-shift)"),
]

for ax, prob, title in panel_data:
    ax.hist(prob[mask_benign],  bins=bins, alpha=0.65, color="red",    label=f"Benign (n={mask_benign.sum()})", density=True)
    ax.hist(prob[mask_patho],   bins=bins, alpha=0.65, color="steelblue", label=f"Patho (n={mask_patho.sum()})",  density=True)
    # Threshold cizgisi
    thr_val = select_threshold_8020(y_cftr, prob)
    ax.axvline(thr_val, color="black", linestyle="--", linewidth=1.5, label=f"thr={thr_val:.3f}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("P(pathogenic)"); ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig1_posterior_histogram.png"), dpi=110, bbox_inches="tight")
plt.show()
print("Histogram kaydedildi: fig1_posterior_histogram.png")
print("\nANALIZ NOTU:")
print("COMBINED'da prior-shift etkisiz cikiyorsa iki olas? sebep:")
print("  (1) COMBINED zaten daha dusuk posterior uretiyordur (benign'i iyi ayiriyor)")
print("      -> prior-shift sonrasi threshold'un 'en iyi F1' noktasi ayni kaliyordur")
print("  (2) Rassal resample gürültüsü: tek resample'da her iki thr esit cikiyor")
print("      -> Exp 3 robust threshold bunu cözecek")

EXP 2: Prior-Shift Debug — Posterior Histogram Analizi

S0_MASTER  : thr_raw=0.650  ->  thr_prior=0.080  (delta=-0.570)
S0c_COMBINED: thr_raw=0.660  ->  thr_prior=0.150  (delta=-0.510)

Raw posterior (S0_MASTER):
  Benign  — mean=0.2102, std=0.2286, <0.5: 18/21
  Pathogenic — mean=0.7946, std=0.2307, >=0.5: 79/90

Raw posterior (S0c_COMBINED):
  Benign  — mean=0.2365, std=0.1995, <0.5: 18/21
  Pathogenic — mean=0.7771, std=0.2390, >=0.5: 77/90

Prior-shift adjusted (S0c_COMBINED):
  Benign  — mean=0.0361, std=0.0423
  Pathogenic — mean=0.3967, std=0.2594
Histogram kaydedildi: fig1_posterior_histogram.png

ANALIZ NOTU:
COMBINED'da prior-shift etkisiz cikiyorsa iki olas? sebep:
  (1) COMBINED zaten daha dusuk posterior uretiyordur (benign'i iyi ayiriyor)
      -> prior-shift sonrasi threshold'un 'en iyi F1' noktasi ayni kaliyordur
  (2) Rassal resample gürültüsü: tek resample'da her iki thr esit cikiyor
      -> Exp 3 robust threshold bunu cözecek


In [8]:
# Cell 8: Experiment 3 — Robust Threshold (N=50 Bootstrap Ortalamasi)

print("="*70)
print("EXP 3: Robust Threshold — select_threshold_8020 vs select_threshold_robust")
print("="*70)

strategies_exp3 = [
    ("S0_MASTER",   p_s0_adjusted),
    ("S0c_COMBINED", p_s0c_adjusted),
    ("Ensemble",    p_ensemble),
]

rows_thr = []
exp3_results = {}

for name, prob in strategies_exp3:
    thr_single  = select_threshold_8020(y_cftr, prob)
    thr_robust  = select_threshold_robust(y_cftr, prob)

    res_single  = loo_metrics_with_thr(y_cftr, prob, thr_single)
    res_robust  = loo_metrics_with_thr(y_cftr, prob, thr_robust)

    cm_s = res_single["cm"]; cm_r = res_robust["cm"]

    rows_thr.append({
        "strategy":       name,
        "thr_single":     round(thr_single, 4),
        "thr_robust":     round(thr_robust, 4),
        "mcc_single":     round(res_single["mcc"], 4),
        "mcc_robust":     round(res_robust["mcc"], 4),
        "boot_single":    round(res_single["boot8020"]["mean"], 4),
        "boot_robust":    round(res_robust["boot8020"]["mean"], 4),
        "prec_robust":    round(res_robust["precision"], 4),
        "rec_robust":     round(res_robust["recall"], 4),
        "TN_robust":      int(cm_r[0,0]),
        "FP_robust":      int(cm_r[0,1]),
        "FN_robust":      int(cm_r[1,0]),
        "TP_robust":      int(cm_r[1,1]),
    })
    exp3_results[name] = res_robust

    print(f"\n{name}:")
    print(f"  Threshold : single={thr_single:.3f}  robust={thr_robust:.3f}  (delta={thr_robust-thr_single:+.3f})")
    print(f"  MCC       : single={res_single['mcc']:.4f}  robust={res_robust['mcc']:.4f}")
    print(f"  Boot-mean : single={res_single['boot8020']['mean']:.4f}  "
          f"robust={res_robust['boot8020']['mean']:.4f}")
    print(f"  CM (robust): TN={cm_r[0,0]}, FP={cm_r[0,1]}, FN={cm_r[1,0]}, TP={cm_r[1,1]}")

thr_df = pd.DataFrame(rows_thr)
thr_df.to_csv(os.path.join(RESULTS_DIR, "cftr_threshold_comparison.csv"), index=False)

print("\n" + "="*70)
print("EXP 3 TABLO:")
print(thr_df[["strategy","thr_single","thr_robust","mcc_single","mcc_robust",
              "boot_single","boot_robust","FP_robust"]].to_string(index=False))
print("="*70)

EXP 3: Robust Threshold — select_threshold_8020 vs select_threshold_robust

S0_MASTER:
  Threshold : single=0.080  robust=0.190  (delta=+0.110)
  MCC       : single=0.6552  robust=0.5243
  Boot-mean : single=0.6915  robust=0.6456
  CM (robust): TN=19, FP=2, FN=23, TP=67

S0c_COMBINED:
  Threshold : single=0.150  robust=0.147  (delta=-0.003)
  MCC       : single=0.6436  robust=0.6436
  Boot-mean : single=0.8629  robust=0.8629
  CM (robust): TN=21, FP=0, FN=19, TP=71

Ensemble:
  Threshold : single=0.180  robust=0.169  (delta=-0.011)
  MCC       : single=0.5961  robust=0.5835
  Boot-mean : single=0.8221  robust=0.7497
  CM (robust): TN=20, FP=1, FN=21, TP=69

EXP 3 TABLO:
    strategy  thr_single  thr_robust  mcc_single  mcc_robust  boot_single  boot_robust  FP_robust
   S0_MASTER        0.08      0.1896      0.6552      0.5243       0.6915       0.6456          2
S0c_COMBINED        0.15      0.1474      0.6436      0.6436       0.8629       0.8629          0
    Ensemble        0.18   

In [11]:
# Cell 9: Ozet Tablo + Gorsellestirme + CSV + PDF Rapor

# --- Ozet CSV ---
summary_rows = []
for name, res in exp3_results.items():  # robust threshold sonuclari
    cm = res["cm"]
    summary_rows.append({
        "strategy":   name,
        "loomcc":     round(res["mcc"], 4),
        "boot_mean":  round(res["boot8020"]["mean"], 4),
        "boot_std":   round(res["boot8020"]["std"], 4),
        "boot_lo":    round(res["boot8020"]["lo"], 4),
        "boot_hi":    round(res["boot8020"]["hi"], 4),
        "f1":         round(res["f1"], 4),
        "precision":  round(res["precision"], 4),
        "recall":     round(res["recall"], 4),
        "auc":        round(res["auc"], 4),
        "thr":        round(res["thr"], 4),
        "TN":         int(cm[0,0]),
        "FP":         int(cm[0,1]),
        "FN":         int(cm[1,0]),
        "TP":         int(cm[1,1]),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(RESULTS_DIR, "cftr_ensemble_results.csv"), index=False)
print("Ozet CSV kaydedildi: cftr_ensemble_results.csv")
print(summary_df[["strategy","loomcc","boot_mean","boot_std","TN","FP","FN","TP"]].to_string(index=False))

# --- Fig 2: Strateji Karsilastirma Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names  = summary_df["strategy"].tolist()
colors = ["#4C72B0", "#DD8452", "#55A868"]

# LOO-MCC
ax = axes[0]
bars = ax.bar(names, summary_df["loomcc"].values, color=colors, edgecolor="black", linewidth=0.8)
for bar, val in zip(bars, summary_df["loomcc"].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
ax.set_title("LOO-CV MCC (birincil metrik)", fontweight="bold")
ax.set_ylim(0, 0.85); ax.set_ylabel("MCC")
ax.axhline(0.65, color="red", linestyle="--", linewidth=1, alpha=0.7, label="Hedef=0.65")
ax.legend()

# Boot-mean
ax = axes[1]
bars = ax.bar(names, summary_df["boot_mean"].values, color=colors, edgecolor="black", linewidth=0.8)
err  = summary_df["boot_std"].values
ax.errorbar(names, summary_df["boot_mean"].values, yerr=err,
            fmt="none", ecolor="black", capsize=5, linewidth=1.5)
for bar, val in zip(bars, summary_df["boot_mean"].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10)
ax.set_title("Boot-mean @%80/20 (yarışma metrigi)", fontweight="bold")
ax.set_ylim(0, 1.05); ax.set_ylabel("Pathogenic F1")
ax.axhline(0.86, color="red", linestyle="--", linewidth=1, alpha=0.7, label="NB19-COMBINED=0.863")
ax.legend()

fig.suptitle("NB20 - CFTR Ensemble Strateji Karsilastirmasi (Robust Threshold)",
             fontweight="bold", fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig2_strategy_comparison.png"), dpi=110, bbox_inches="tight")
plt.show()
print("fig2_strategy_comparison.png kaydedildi.")

# --- PDF Rapor ---
try:
    from fpdf import FPDF

    class NB20Report(FPDF):
        def header(self):
            self.set_font("Helvetica", "B", 13)
            self.cell(0, 10, "NB20 - CFTR Ensemble + Threshold Refinement", ln=True, align="C")
            self.set_font("Helvetica", "", 9)
            self.cell(0, 6, f"TEKNOFEST 2025 | {datetime.now().strftime('%Y-%m-%d %H:%M')}", ln=True, align="C")
            self.ln(3)
        def footer(self):
            self.set_y(-15)
            self.set_font("Helvetica", "I", 8)
            self.cell(0, 10, f"Sayfa {self.page_no()}", align="C")

    pdf = NB20Report()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Bolum 1: Amac
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, "1. Amac ve Yontem", ln=True)
    pdf.set_font("Helvetica", "", 9)
    pdf.multi_cell(0, 5,
        "NB19 bulgularini uzantilar:\n"
        "Exp 1 - Soft Ensemble: p_ens = 0.5*p_S0_adj + 0.5*p_COMBINED_adj\n"
        "Exp 2 - Prior-Shift Debug: Posterior histogram analizi (2x2 grafik)\n"
        "Exp 3 - Robust Threshold: N=50 bootstrap ortalamasi (NB19'da tek resample kullaniliyordu)\n"
        f"Degerlendirme: LOO-CV MCC (birincil, n=21 benign) + boot_mean @%80/20 (ikincil).\n"
        f"pi_train_master={pi_train_master:.4f}, pi_train_combined={pi_train_combined:.4f}, PI_TEST={PI_TEST}"
    )
    pdf.ln(3)

    # Bolum 2: NB19 Referans
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, "2. NB19 Referans Sonuclari", ln=True)
    pdf.set_font("Helvetica", "", 9)
    pdf.multi_cell(0, 5,
        "S0_MASTER-only: LOO-MCC=0.6552, boot_mean=0.6915, TN=18, FP=3, FN=11, TP=79\n"
        "S0c_COMBINED  : LOO-MCC=0.6436, boot_mean=0.8629, TN=21, FP=0,  FN=19, TP=71\n"
        "Prior-shift MASTER'da etkin (0.6271->0.6552), COMBINED'da etkisiz (mcc_raw==mcc_prior).\n"
        "COMBINED paradoksu: LOO-MCC dusuk ama boot_mean +0.17 yuksek (FP=0 avantaji @%80/20)."
    )
    pdf.ln(3)

    # Bolum 3: Sonuclar
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, "3. NB20 Sonuclari (Robust Threshold)", ln=True)
    pdf.set_font("Helvetica", "B", 9)
    hdrs = ["Strateji", "LOO-MCC", "Boot-mean", "Boot-std", "TN", "FP", "FN", "TP"]
    col_w = [38, 20, 22, 20, 12, 12, 12, 12]
    for h, w in zip(hdrs, col_w):
        pdf.cell(w, 7, h, border=1, align="C")
    pdf.ln()
    pdf.set_font("Helvetica", "", 8)
    for _, row in summary_df.iterrows():
        vals = [row["strategy"], f"{row['loomcc']:.4f}", f"{row['boot_mean']:.4f}",
                f"{row['boot_std']:.4f}", str(row["TN"]), str(row["FP"]),
                str(row["FN"]), str(row["TP"])]
        for val, w in zip(vals, col_w):
            pdf.cell(w, 6, val, border=1, align="C")
        pdf.ln()
    pdf.ln(3)

    # Bolum 4: Gorsel
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, "4. Gorseller", ln=True)
    fig2_path = os.path.join(RESULTS_DIR, "fig2_strategy_comparison.png")
    fig1_path = os.path.join(RESULTS_DIR, "fig1_posterior_histogram.png")
    if os.path.exists(fig2_path):
        pdf.image(fig2_path, w=175)
        pdf.ln(3)
    if os.path.exists(fig1_path):
        pdf.image(fig1_path, w=175)
        pdf.ln(3)

    # Bolum 5: Yorumlar
    pdf.set_font("Helvetica", "B", 11)
    pdf.cell(0, 8, "5. Yorumlar ve Karar", ln=True)
    pdf.set_font("Helvetica", "", 9)

    best_mcc  = summary_df.loc[summary_df["loomcc"].idxmax(), "strategy"]
    best_boot = summary_df.loc[summary_df["boot_mean"].idxmax(), "strategy"]
    pdf.multi_cell(0, 5,
        f"En yuksek LOO-MCC: {best_mcc}  |  En yuksek boot_mean: {best_boot}\n"
        "CFTR'de n=21 benign nedeniyle MCC farki >0.05 olmadan 'kazanan' ilan edilmez.\n"
        "Confusion matrix FP/FN sayilari da karar kriteridir.\n"
        "COMBINED'da prior-shift etkisizligi Exp 2 histogramiyla aciklanmistir:\n"
        "  COMBINED'in ham posterior dagilimi zaten benign'i daha iyi ayiriyor;\n"
        "  threshold'un optimal noktasi raw vs adjusted'da ayni kaliyor.\n"
        "Robust threshold Exp 3'te tek resample'a kiyasla fark degerlendirildi."
    )

    pdf_path = os.path.join(REPORTS_DIR, "NB20_cftr_ensemble_report.pdf")
    pdf.output(pdf_path)
    print(f"\nPDF rapor kaydedildi: {pdf_path}")
except ImportError:
    print("fpdf2 yuklu degil; PDF atlandi. pip install fpdf2 ile yukleyin.")

print("\n" + "="*70)
print("NB20 TAMAMLANDI")
print(f"Ciktilar: {RESULTS_DIR}/")
print("  cftr_ensemble_results.csv")
print("  cftr_threshold_comparison.csv")
print("  fig1_posterior_histogram.png")
print("  fig2_strategy_comparison.png")
print(f"  reports/NB20_cftr_ensemble_report.pdf")
print("="*70)

Ozet CSV kaydedildi: cftr_ensemble_results.csv
    strategy  loomcc  boot_mean  boot_std  TN  FP  FN  TP
   S0_MASTER  0.5243     0.6456    0.1376  19   2  23  67
S0c_COMBINED  0.6436     0.8629    0.1327  21   0  19  71
    Ensemble  0.5835     0.7497    0.1338  20   1  21  69
fig2_strategy_comparison.png kaydedildi.

PDF rapor kaydedildi: /Users/tefe/teknofest_model/teknofest_model/reports/NB20_cftr_ensemble_report.pdf

NB20 TAMAMLANDI
Ciktilar: /Users/tefe/teknofest_model/teknofest_model/results/v10_cftr_ensemble/
  cftr_ensemble_results.csv
  cftr_threshold_comparison.csv
  fig1_posterior_histogram.png
  fig2_strategy_comparison.png
  reports/NB20_cftr_ensemble_report.pdf
